# Pandas Analysis

This notebook performs data analysis using Pandas on the cleaned movie datasets.

The analysis includes:
- Loading data from MySQL
- Data filtering and sorting
- Aggregation and grouping
- Calculating statistical measures
- Combining related datasets
- Identifying patterns and trends
- Extracting meaningful insights from the data

In [2]:
import pymysql
import pandas as pd

connection = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="TMBD_analytics"
)



In [4]:
genres_df = pd.read_sql("SELECT * FROM genres", connection)
cast_df = pd.read_sql("SELECT * FROM cast", connection)
crew_df = pd.read_sql("SELECT * FROM crew", connection)
keywords_df = pd.read_sql("SELECT * FROM keywords", connection)
movies_df = pd.read_sql("SELECT * FROM movies", connection)
movie_genres_df = pd.read_sql("SELECT * FROM movie_genres", connection)

C:\Users\Gokul\AppData\Local\Temp\ipykernel_3964\3755885773.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  genres_df = pd.read_sql("SELECT * FROM genres", connection)
C:\Users\Gokul\AppData\Local\Temp\ipykernel_3964\3755885773.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cast_df = pd.read_sql("SELECT * FROM cast", connection)
C:\Users\Gokul\AppData\Local\Temp\ipykernel_3964\3755885773.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  crew_df = pd.read_sql("SELECT * FROM crew", connection)
C:\Users\Gokul\AppDat

#### Q1. List the total number of unique movies, genres, cast members and crew members collected across all tables.

In [5]:
print("Unique Movies:", movies_df["movie_id"].nunique())
print("Unique Genres:", genres_df["genre_id"].nunique())
print("Unique Cast Members:", cast_df["person_id"].nunique())
print("Unique Crew Members:", crew_df["person_id"].nunique())

Unique Movies: 2503
Unique Genres: 19
Unique Cast Members: 11300
Unique Crew Members: 2848


*Insight:
The dataset has many unique movies, genres, actors, and crew members.*

#### Q2. List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.

In [6]:
invalid_financial = movies_df[
    (movies_df["budget"].isna()) |
    (movies_df["budget"] == 0) |
    (movies_df["revenue"].isna()) |
    (movies_df["revenue"] == 0)
]

invalid_financial[
    ["movie_id", "title", "budget", "revenue"]
]

,movie_id,title,budget,revenue
66,147,The 400 Blows,0.0,0.0
158,422,8½,0.0,0.0
253,653,Nosferatu,0.0,27964.0
314,797,Persona,0.0,250000.0
334,832,M,0.0,35274.0
...,...,...,...,...
2427,868759,Ghosted,40000000.0,0.0
2433,899082,Harry Potter 20th Anniversary: Return to Hogwarts,0.0,0.0
2457,950396,The Gorge,70000000.0,0.0
2467,1005331,Carry-On,47000000.0,0.0


*Insight:
Some movies have missing or zero budget or revenue. We should exclude them from financial analysis.*

#### Q3. List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.

In [7]:
valid_movies = movies_df[
    (movies_df["budget"] > 0) &
    (movies_df["revenue"] > 0)
].copy()

valid_movies["profit"] = (
    valid_movies["revenue"] - valid_movies["budget"]
)

valid_movies["roi"] = (
    valid_movies["profit"] / valid_movies["budget"]
)

valid_movies[
    ["movie_id", "title", "budget", "revenue", "profit", "roi"]
]

,movie_id,title,budget,revenue,profit,roi
0,5,Four Rooms,4000000.0,4257350.0,257350.0,0.064338
1,11,Star Wars,11000000.0,775398000.0,764398000.0,69.490727
2,12,Finding Nemo,94000000.0,940336000.0,846336000.0,9.003574
3,13,Forrest Gump,55000000.0,677388000.0,622388000.0,11.316145
4,14,American Beauty,15000000.0,356297000.0,341297000.0,22.753133
...,...,...,...,...,...,...
2498,1368166,The Housemaid,35000000.0,399645000.0,364645000.0,10.418429
2499,1368167,Date Night,93214800.0,87595700.0,-5619100.0,-0.060281
2500,1368168,Transformers,110099000.0,274747000.0,164648000.0,1.495454
2501,1368169,The SpongeBob Movie: Sponge Out of Water,39284900.0,570658000.0,531373100.0,13.526141


*Insight:
Profit shows the money earned after the budget. ROI shows the return from the investment.*

#### Q4. List the top 15 movies by return on investment, considering only movies with a budget of at least $1 million.

In [8]:
top_15_roi = valid_movies[
    valid_movies["budget"] >= 1_000_000
].sort_values(
    "roi",
    ascending=False
).head(15)

top_15_roi[
    ["title", "budget", "revenue", "profit", "roi"]
]

,title,budget,revenue,profit,roi
2191,Dragon Ball Super: Broly,1000000.0,125003000.0,124003000.0,124.003000
152,Snow White and the Seven Dwarfs,1488420.0,184925000.0,183436580.0,123.242485
1211,The Rocky Horror Picture Show,1400000.0,171181000.0,169781000.0,121.272143
412,Rocky,1000000.0,117253000.0,116253000.0,116.253000
300,Gone with the Wind,4000000.0,402353000.0,398353000.0,99.588250
794,The Jungle Book,4000000.0,378000000.0,374000000.0,93.500000
1001,Cinderella,2900000.0,263600000.0,260700000.0,89.896552
78,Saw,1200000.0,104046000.0,102846000.0,85.705000
1055,One Hundred and One Dalmatians,3600000.0,303000000.0,299400000.0,83.166667
221,E.T. the Extra-Terrestrial,10500000.0,797307000.0,786807000.0,74.934000


*Insight:
These 15 movies have the highest ROI among movies with a budget of at least $1 million.*

#### Q5. List the number of movies released per year, and identify which year had the highest movie output in the dataset

In [9]:
movies_df["release_date"] = pd.to_datetime(
    movies_df["release_date"],
    errors="coerce"
)

movies_per_year = (
    movies_df
    .dropna(subset=["release_date"])
    .assign(
        release_year=lambda x: x["release_date"].dt.year
    )
    .groupby("release_year")
    .size()
    .reset_index(name="movie_count")
    .sort_values("release_year")
)

movies_per_year

,release_year,movie_count
0,1921,1
1,1922,1
2,1927,1
3,1931,2
4,1936,1
...,...,...
84,2022,65
85,2023,62
86,2024,43
87,2025,26


*Insight:
The year with the most movies had the highest movie output.*

#### Q6. List all movies whose runtime is a statistical outlier compared to the rest of the dataset.

In [10]:
Q1 = movies_df["runtime"].quantile(0.25)
Q3 = movies_df["runtime"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

runtime_outliers = movies_df[
    (movies_df["runtime"] < lower_limit) |
    (movies_df["runtime"] > upper_limit)
]

runtime_outliers[
    ["movie_id", "title", "runtime"]
]

,movie_id,title,runtime
50,111,Scarface,170
56,120,The Lord of the Rings: The Fellowship of the Ring,179
57,121,The Lord of the Rings: The Two Towers,179
58,122,The Lord of the Rings: The Return of the King,201
88,197,Braveheart,178
98,238,The Godfather,175
100,240,The Godfather Part II,202
107,254,King Kong,188
118,285,Pirates of the Caribbean: At World's End,169
121,297,Meet Joe Black,178


*Insight:
Some movies have unusually short or long runtimes.*

#### Q7. List each movie's primary genre and compare average popularity across primary genres.

In [11]:
primary_genre = (
    movie_genres_df
    .drop_duplicates("movie_id")
    .merge(
        genres_df,
        on="genre_id",
        how="left"
    )
    .merge(
        movies_df[
            ["movie_id", "title", "popularity"]
        ],
        on="movie_id",
        how="left"
    )
)

primary_genre[
    ["movie_id", "title", "genre_name", "popularity"]
]

,movie_id,title,genre_name,popularity
0,157336,Interstellar,Adventure,73.0452
1,27205,Inception,Action,55.8497
2,24428,The Avengers,Science Fiction,72.2515
3,155,The Dark Knight,Action,60.5510
4,19995,Avatar,Action,48.1996
...,...,...,...,...
2494,10664,[REC]²,Thriller,5.7505
2495,1114513,Speak No Evil,Horror,13.8268
2496,13680,The Game Plan,Comedy,7.7220
2497,7461,Vantage Point,Drama,6.9261


In [12]:
avg_popularity = (
    primary_genre
    .groupby("genre_name")["popularity"]
    .mean()
    .reset_index(name="average_popularity")
    .sort_values(
        "average_popularity",
        ascending=False
    )
)

avg_popularity

,genre_name,average_popularity
14,Science Fiction,21.261837
11,Music,18.324360
2,Animation,16.811212
7,Family,16.710635
0,Action,16.385190
16,War,15.470317
10,Horror,14.851833
1,Adventure,14.785510
8,Fantasy,13.022920
12,Mystery,12.427131


*Insight:
Some genres have higher average popularity than other genres.*

#### Q8. List the top 10 most prolific actors by number of movies, along with their average movie rating.

In [13]:
actor_analysis = (
    cast_df
    .merge(
        movies_df[
            ["movie_id", "vote_average"]
        ],
        on="movie_id",
        how="left"
    )
    .groupby(
        ["person_id", "actor_name"]
    )
    .agg(
        movie_count=("movie_id", "nunique"),
        average_rating=("vote_average", "mean")
    )
    .reset_index()
    .sort_values(
        "movie_count",
        ascending=False
    )
    .head(10)
)

actor_analysis

,person_id,actor_name,movie_count,average_rating
679,2231,Samuel L. Jackson,39,6.938692
106,287,Brad Pitt,38,7.257289
127,380,Robert De Niro,38,7.228316
37,85,Johnny Depp,37,6.856973
13,31,Tom Hanks,33,7.289758
1272,5293,Willem Dafoe,32,7.114781
413,1245,Scarlett Johansson,32,7.056344
2674,13240,Mark Wahlberg,32,6.630063
161,500,Tom Cruise,31,7.040613
64,192,Morgan Freeman,31,6.963742


*Insight:
These are the top 10 actors who appeared in the most movies.*

#### Q9. List any movie titles that appear more than once, along with their release years.

In [14]:
duplicate_titles = movies_df[
    movies_df["title"].duplicated(
        keep=False
    )
].copy()

duplicate_titles["release_year"] = (
    duplicate_titles["release_date"].dt.year
)

duplicate_titles[
    ["title", "release_year"]
].sort_values("title")

,title,release_year
140,A Nightmare on Elm Street,1984
1164,A Nightmare on Elm Street,2010
323,Aladdin,1992
2068,Aladdin,2019
1042,Alice in Wonderland,1951
...,...,...
1368,The Thing,2011
346,Total Recall,1990
1381,Total Recall,2012
489,Transformers,2007


*Insight:
Some movie titles appear more than once. They may be duplicates or remakes.*

#### Q10. List movies whose popularity score and vote count don't follow the general pattern.

In [19]:
movies_check = movies_df[
    ["movie_id", "title", "popularity", "vote_count"]
].dropna().copy()

# High popularity but low vote count
unusual_movies = movies_check[
    (movies_check["popularity"] > movies_check["popularity"].quantile(0.90)) &
    (movies_check["vote_count"] < movies_check["vote_count"].quantile(0.25))
]

unusual_movies[
    ["title", "popularity", "vote_count"]
].sort_values("popularity", ascending=False)


,title,popularity,vote_count
2495,Disclosure Day,394.3350,2328
2479,Backrooms,258.4740,2483
2444,Mortal Kombat II,99.5231,2260
2496,Lee Cronin's The Mummy,79.4601,2430
2498,The Housemaid,41.4337,2789
2482,How to Train Your Dragon,35.4872,3008
2288,Final Destination Bloodlines,34.1109,3001
2267,Lilo & Stitch,33.2161,2228
2290,Mission: Impossible - The Final Reckoning,31.1708,3001
316,Lolita,27.5391,2271


*Insight:Some movies have high popularity but relatively low vote counts*